# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 任務分解

任務分解是規劃設計模式的核心。與其讓單一代理人處理一個複雜的請求直到完成，
我們將問題拆解成更小且定義明確的<strong>子任務</strong>。
每個子任務會指派給專門的代理人（例如，航班、酒店、活動），並具有明確的
優先順序和依賴關係排序。

這種方法具備以下幾項好處：
- <strong>清晰度</strong>：每個子任務負責單一責任。
- <strong>並行性</strong>：獨立的子任務可以同時執行。
- <strong>可靠性</strong>：故障被限制於個別子任務。
- <strong>預算追蹤</strong>：成本依子任務估算並匯總。


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## 建立帶有結構化輸出的規劃代理

規劃代理作為<strong>前台協調員</strong>。根據高層的旅行請求，
它會產生一個結構化的 `TravelPlan` — 將請求分解為子任務、設定優先順序，
並識別依賴關係，以便禮賓司或執行層能夠執行工作。


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## 使用專家工具執行計劃

當前台代理產生了結構化計劃後，<strong>禮賓代理</strong>會執行該計劃。
每個專家工具處理一類子任務（航班、酒店、活動）。禮賓
會依照計劃中子任務的依賴順序逐一遍歷，並將每個子任務分派給
適當的工具。


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## 摘要

在這課中，你學習了 AI 代理的 <strong>規劃設計模式</strong>：

1. <strong>任務分解</strong> — 前台規劃代理將複雜的旅遊請求拆解成
   使用 Pydantic 模型的結構化子任務，並為每個子任務分配專家代理，設置優先級
   和依賴關係。
2. <strong>結構化輸出</strong> — 通過傳遞 `response_format`，代理返回經過驗證的
   `TravelPlan` 對象，而非自由格式的文字，使後續處理更可靠。
3. <strong>計劃執行</strong> — 禮賓代理使用專家工具反覆執行子任務
   （`book_flight`、`reserve_hotel`、`book_activity`）來執行計劃並回報結果。

此模式將 <em>要做什麼</em>（規劃）與 <em>怎麼做</em>（執行）分開，使代理
更模組化、易於測試且更易擴展。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
